A **Random Forest** is an ensemble machine learning model that combines the predictions of multiple individual decision trees to create a highly accurate and stable output.

# Terminologies

**Ensemble Learning** : An umbrella term in machine learning where you combine the predictions of multiple individual models (like a group of separate decision trees) to create a single, more accurate, and more stable final answer.



**Weak Learner** : A base machine learning model that is individually simple and prone to high error rates (like a shallow decision tree). A Random Forest takes hundreds of these "weak learners" and combines them into one highly accurate "Strong Learner."



**Variance vs. Bias** : 
- Variance represents a model's tendency to overfit and change wildly based on minor shifts in the training data. Single decision trees suffer from exceptionally high variance.
- Bias represents structural blindness where a model is too simple to capture real patterns (underfitting).
- The Magic of Random Forests: They dramatically lower the variance of your overall system without increasing its bias.



**Aggregation**: The mathematical step of combining the results from all your trees. For Classification Forest problems (e.g., Cat vs. Dog), it aggregates using a Majority Vote (hard voting). For Regression Forest problems (e.g., predicting continuous lifespans), it aggregates by taking the Statistical Mean (average) of all tree outputs.



**Bootstrapping (Sampling with Replacement):** Create multiple new training datasets from your original data by picking rows at random with replacement. Some rows may be chosen more than once, while others are left out.



**Bagging (Bootstrap Aggregating):** It is a two-step combo: first you Bootstrap your data to create unique row variations, and then you Aggregate (average/vote) all the results together at the end.



**Out-Of-Bag (OOB) Data:** The leftover rows that were completely skipped and never selected during a tree's specific bootstrapping process. Because a tree has never seen its OOB rows during training, you can use these leftovers as an automated "validation test set" to see how accurate that tree is without needing a separate cross-validation split.



**Feature Bagging (Subspace Sampling):** Forcing each split junction (node) inside a tree to randomly pick its next rule from a restricted, blindfolded pool of features (typically \(k = \sqrt{d}\)), ensuring that no two trees look at the exact same column structures.



**n_estimators (M):** The total count of individual decision trees you choose to grow inside your forest. A standard value is usually between 100 and 500 trees. Adding more trees never causes overfitting; it simply makes the final voting pool more stable.


**max_features (k):** The maximum number of random features a single node is allowed to evaluate during a split search.


**Bootstrap (True/False):** A switch determining whether the model should shuffle data rows using replacement (True) or train every single tree on the exact same original dataset rows (False).

 
**MDI (Mean Decrease in Impurity) / Gini Importance:** The method used to compute Feature Importance across a forest. It tracks every time a specific feature is selected across all nodes in all trees, calculates how much Gini impurity or variance dropped because of that specific split, and adds those scores together. Features with a high MDI are the "VIP features" driving your model.

# Classification

## Mathematical Example

1. **Setup & Dataset**
    
    Given dataset $D = \{(x_1, y_1), (x_2, y_2), (x_3, y_3), (x_4, y_4)\} \subset \mathbb{R}^4 \times \{0, 1\}$:
    $$X = \begin{bmatrix}  x_{1,1} & x_{1,2} & x_{1,3} & x_{1,4} \\ x_{2,1} & x_{2,2} & x_{2,3} & x_{2,4} \\ x_{3,1} & x_{3,2} & x_{3,3} & x_{3,4} \\ x_{4,1} & x_{4,2} & x_{4,3} & x_{4,4} \end{bmatrix} = \begin{bmatrix}  2.0 & 5.0 & 1.0 & 0.5 \\ 7.0 & 1.0 & 4.0 & 2.5 \\ 8.0 & 4.0 & 3.0 & 1.5 \\ 1.0 & 8.0 & 2.0 & 0.1 \end{bmatrix}, \quad  \mathbf{y} = \begin{bmatrix} 0 \\ 1 \\ 1 \\ 0 \end{bmatrix}$$
    
    - Parameters: $M = 3$ trees, $k = \lfloor\sqrt{d}\rfloor = \lfloor\sqrt{4}\rfloor = 2$ features per split.
    - Impurity measure: $G(S) = 1 - \sum_{c \in \{0,1\}} p_c^2$
    - Information Gain: $\Delta G(S, j, t) = G(S) - \left( \frac{\vert{}S_L\vert{}}{\vert{}S\vert{}} G(S_L) + \frac{\vert{}S_R\vert{}}{\vert{}S\vert{}} G(S_R) \right)$

2. **Mathematical Construction of Forest $\mathcal{F} = \{T_1, T_2, T_3\}$**
    
    **Tree $T_1$**
    
    1. **Bootstrap Resampling:**
        
        $$I_1 = (1, 2, 1, 4) \implies D_1 = \left\{ \begin{pmatrix} 2.0, 5.0, 1.0, 0.5 \end{pmatrix} \to 0, \; \begin{pmatrix} 7.0, 1.0, 4.0, 2.5 \end{pmatrix} \to 1, \; \begin{pmatrix} 2.0, 5.0, 1.0, 0.5 \end{pmatrix} \to 0, \; \begin{pmatrix} 1.0, 8.0, 2.0, 0.1 \end{pmatrix} \to 0 \right\}$$
        
        $$\mathbf{y}_{D_1} = [0, 1, 0, 0]^T \implies p_0 = \frac{3}{4}, \; p_1 = \frac{1}{4}$$
        
        $$G(D_1) = 1 - \left( \left(\frac{3}{4}\right)^2 + \left(\frac{1}{4}\right)^2 \right) = 1 - \frac{10}{16} = 0.375$$
        
    2. **Feature Subsampling & Split Evaluation**:
        
        Select feature subset $F_1 = \{1, 4\} \subset \{1, 2, 3, 4\}$.
        - Candidate Split 1: $j = 1, t = 1.5$
        
        $$S_L = \{i \mid x_{i,1} \le 1.5\} = \{ \mathbf{y}=[0] \}, \quad S_R = \{i \mid x_{i,1} > 1.5\} = \{ \mathbf{y}=[0, 1, 0] \}$$
        $$G(S_L) = 0.0, \quad G(S_R) = 1 - \left( \left(\frac{2}{3}\right)^2 + \left(\frac{1}{3}\right)^2 \right) = \frac{4}{9} \approx 0.444$$
        
        $$\Delta G(D_1, 1, 1.5) = 0.375 - \left( \frac{1}{4}(0.0) + \frac{3}{4}\left(\frac{4}{9}\right) \right) = 0.375 - 0.333 = 0.042$$
        
        - Candidate Split 2: $j = 4, t = 1.5$
        
        $$S_L = \{i \mid x_{i,4} \le 1.5\} = \{ \mathbf{y}=[0, 0, 0] \}, \quad S_R = \{i \mid x_{i,4} > 1.5\} = \{ \mathbf{y}=[1] \}$$
        $$G(S_L) = 0.0, \quad G(S_R) = 0.0$$
        $$\Delta G(D_1, 4, 1.5) = 0.375 - \left( \frac{3}{4}(0.0) + \frac{1}{4}(0.0) \right) = \mathbf{0.375}$$
        
    3. **Optimal Split Strategy**:
        $$(j^*, t^*) = \arg\max_{(j,t)} \Delta G = (4, 1.5)$$
        $$T_1(\mathbf{x}) = \begin{cases} 0 & \text{if } x_4 \le 1.5 \\ 1 & \text{if } x_4 > 1.5 \end{cases}$$
        
    **Tree $T_2$**
    
    1. **Bootstrap Resampling**:
    $$I_2 = (2, 3, 3, 4) \implies \mathbf{y}_{D_2} = [1, 1, 1, 0]^T$$
    $$G(D_2) = 1 - \left( \left(\frac{1}{4}\right)^2 + \left(\frac{3}{4}\right)^2 \right) = 0.375$$
    
    2. **Feature Subsampling & Split Evaluation**:
    Select feature subset $F_2 = \{2, 3\}$.
    - Candidate Split:  $j = 2, t = 2.5$:
    
    $$S_L = \{y=1\}, \quad S_R = \{y=1, 1, 0\} \implies G(S_R) = 1 - \left(\left(\frac{2}{3}\right)^2 + \left(\frac{1}{3}\right)^2\right) = \frac{4}{9}$$
    $$\Delta G = 0.375 - \left(\frac{1}{4}(0.0) + \frac{3}{4}\left(\frac{4}{9}\right)\right) = 0.375 - 0.333 = 0.042$$

    - Candidate Split:  $j = 2, t = 6.0$:
    $$S_L = \{y=1, 1, 1\}, \quad S_R = \{y=0\} \implies G(S_L) = 0.0, \; G(S_R) = 0.0$$
    $$\Delta G(D_2, 2, 6.0) = 0.375 - 0.0 = \mathbf{0.375}$$

    
    3. **Optimal Split Strategy**:
    Both $(j=2, t=6.0)$ and $(j=3, t=2.5)$ achieve maximum gain $\Delta G_{\max} = 0.375$. Breaking tie deterministically by lowest feature index yields:
    $$(j^*, t^*) = (2, 6.0) \implies T_2(\mathbf{x}) = \begin{cases} 1 & \text{if } x_2 \le 6.0 \\ 0 & \text{if } x_2 > 6.0 \end{cases}$$
    
    **Tree $T_3$**

    1. **Bootstrap Resampling**:

    $$I_3 = (1, 2, 3, 4) \implies \mathbf{y}_{D_3} = [0, 1, 1, 0]^T$$
    $$G(D_3) = 1 - \left( \left(\frac{2}{4}\right)^2 + \left(\frac{2}{4}\right)^2 \right) = 0.500$$
    
    2. **Feature Subsampling & Split Evaluation**:
    Select feature subset $F_3 = \{1, 3\}$.
    - Candidate Split $j = 1, t = 1.5$:
    $$S_L = \{y=0\}, \quad S_R = \{y=0, 1, 1\} \implies G(S_R) = \frac{4}{9}$$
    $$\Delta G(D_3, 1, 1.5) = 0.500 - \frac{3}{4}\left(\frac{4}{9}\right) = 0.500 - 0.333 = 0.167$$

    - Candidate Split $j = 3, t = 1.5$:
    $$S_L = \{y=0\}, \quad S_R = \{y=0, 1, 1\} \implies G(S_R) = \frac{4}{9}$$
    $$\Delta G(D_3, 3, 1.5) = 0.500 - \frac{3}{4}\left(\frac{4}{9}\right) = 0.167$$

    
    3. **Optimal Split Strategy:**
    Both $(j=1, t=4.5)$ and $(j=3, t=2.5)$ yield perfect reduction $\Delta G_{\max} = 0.500$. Breaking tie by lowest feature index:
    $$(j^*, t^*) = (1, 4.5) \implies T_3(\mathbf{x}) = \begin{cases} 0 & \text{if } x_1 \le 4.5 \\ 1 & \text{if } x_1 > 4.5 \end{cases}$$
    
3. **Inference / Evaluation on New Vector $\mathbf{x}^*$**
    
    Let $\mathbf{x}^* = [x_1^*, x_2^*, x_3^*, x_4^*]^T = [6.0, 2.0, 3.5, 0.4]^T$.
    Individual Tree Outputs:
    $$T_1(\mathbf{x}^*) \implies x_4^* = 0.4 \le 1.5 \implies y^{(1)} = 0$$
    $$T_2(\mathbf{x}^*) \implies x_2^* = 2.0 \le 6.0 \implies y^{(2)} = 1$$
    $$T_3(\mathbf{x}^*) \implies x_1^* = 6.0 > 4.5 \implies y^{(3)} = 1$$
    
    Aggregation / Majority Vote:
    $$\mathbf{y}_{\text{ensemble}} = \begin{bmatrix} y^{(1)} & y^{(2)} & y^{(3)} \end{bmatrix}^T = \begin{bmatrix} 0 & 1 & 1 \end{bmatrix}^T$$
    $$\hat{y}_{\mathcal{F}}(\mathbf{x}^*) = \text{mode}\left( \{ y^{(m)} \}_{m=1}^3 \right) = \arg\max_{c \in \{0, 1\}} \sum_{m=1}^{3} \mathbb{I}\left(y^{(m)} = c\right)$$
    $$\sum_{m=1}^{3} \mathbb{I}\left(y^{(m)} = 0\right) = 1, \quad \sum_{m=1}^{3} \mathbb{I}\left(y^{(m)} = 1\right) = 2$$
    $$\therefore \hat{y}_{\mathcal{F}}(\mathbf{x}^*) = \mathbf{1}$$

## Random Forest Classifier Algorithm:  (Summary)

1. **Initialization**
    - **Dataset Input**: Take the full training dataset $D$ containing $N$ samples and $d$ total features.
    - **Forest Assembly**: Specify the total number of decision trees to build, $M$ (n_estimators).

2. **Bootstrapping (Bagging)**
    For each tree $m \in \{1, 2, \dots, M\}$:
    - **Row Resampling**: Create a unique dataset $D_m$ by randomly sampling $N$ rows with replacement from $D$.
    - **Duplication & OOB**: Because sampling is with replacement, some rows are duplicated while roughly $36.8\%$ are left out (Out-Of-Bag).

3. **Tree Building with Feature Subsampling**

    Grow each tree on its bootstrapped dataset $D_m$. At every individual node split:
    
    1. **Random Subsampling:** Randomly select a subset of $k$ features from the total $d$ features (where $k = \lfloor\sqrt{d}\rfloor$ for classification).
    
    2. **Threshold Evaluation:** For each of the $k$ selected features:
        - Sort the unique feature values.
        - Calculate midpoints between adjacent sorted values to generate candidate thresholds $t$.
        - Evaluate the split impurity (Gini index or Entropy) for every threshold.
        - Calculate Information Gain: $\Delta G = G_{\text{node}} - G_{\text{split}}$.
    
    3. **Node Split:** Pick the feature $j^*$ and threshold $t^*$ that yield the maximum Information Gain across all evaluated candidates in the subset.
    
    4. **Recursive Step:** Repeat steps 1–3 at subsequent child nodes until stopping criteria are met (e.g., node purity, max depth, or minimum sample size).
    
4. **Inference & Prediction**
    When a new sample vector $\mathbf{x}^*$ is passed to the trained forest:
    1. **Individual Predictions:** Every tree evaluates $\mathbf{x}^*$ and produces its own class prediction:
    $$\hat{y}_m = T_m(\mathbf{x}^*) \quad \text{for } m = 1, 2, \dots, M$$
    2. **Majority Voting:** Aggregate all individual predictions into a vote array:
    $$\mathbf{v} = [\hat{y}_1, \hat{y}_2, \dots, \hat{y}_M]$$
    3. **Final Ensemble Output:** Select the class that receives the majority vote:
    $$\hat{y}_{\text{forest}} = \text{mode}(\mathbf{v}) = \arg\max_{c} \sum_{m=1}^{M} \mathbb{I}(\hat{y}_m = c)$$

## Python Code

In [ ]:
import numpy as np


class DecisionNode:
    """Represents a node or leaf in a decision tree."""

    def __init__(
        self,
        feature=None,
        threshold=None,
        left=None,
        right=None,
        *,
        value=None
    ):
        self.feature = feature  # Index of feature to split on
        self.threshold = threshold  # Threshold value for split
        self.left = left  # Left child node (x <= threshold)
        self.right = right  # Right child node (x > threshold)
        self.value = value  # Predicted class if leaf node

    def is_leaf(self):
        return self.value is not None


class DecisionTreeClassifier:
    """A single Decision Tree with Gini Impurity and Feature Subsampling."""

    def __init__(self, max_depth=10, min_samples_split=2, max_features=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features  # k features to consider per split
        self.root = None

    def _gini(self, y):
        """Calculates Gini Impurity: G = 1 - sum(p_i^2)"""
        if len(y) == 0:
            return 0.0
        p = np.bincount(y) / len(y)
        return 1.0 - np.sum(p**2)

    def _information_gain(self, y, y_left, y_right):
        """Information Gain = G_parent - (w_L * G_left + w_R * G_right)"""
        parent_gini = self._gini(y)
        n = len(y)
        n_l, n_r = len(y_left), len(y_right)

        if n_l == 0 or n_r == 0:
            return 0.0

        child_gini = (n_l / n) * self._gini(y_left) + (n_r / n) * self._gini(
            y_right
        )
        return parent_gini - child_gini

    def _best_split(self, X, y, feat_idxs):
        """Finds the optimal feature and threshold among selected feature indices."""
        best_gain = -1.0
        split_feat, split_thresh = None, None

        for feat in feat_idxs:
            X_column = X[:, feat]
            # Get sorted unique values to compute midpoints
            unique_vals = np.sort(np.unique(X_column))

            # Calculate midpoints as candidate thresholds
            thresholds = (unique_vals[:-1] + unique_vals[1:]) / 2.0

            for thresh in thresholds:
                left_idxs = np.where(X_column <= thresh)[0]
                right_idxs = np.where(X_column > thresh)[0]

                if len(left_idxs) == 0 or len(right_idxs) == 0:
                    continue

                gain = self._information_gain(
                    y, y[left_idxs], y[right_idxs]
                )

                if gain > best_gain:
                    best_gain = gain
                    split_feat = feat
                    split_thresh = thresh

        return split_feat, split_thresh

    def _build_tree(self, X, y, depth=0):
        n_samples, n_features = X.shape
        n_labels = len(np.unique(y))

        # Stopping criteria: pure node, max depth, or insufficient samples
        if (
            depth >= self.max_depth
            or n_labels == 1
            or n_samples < self.min_samples_split
        ):
            leaf_val = np.bincount(y).argmax()
            return DecisionNode(value=leaf_val)

        # Feature Subsampling: select k random features at THIS node
        k = (
            self.max_features
            if self.max_features is not None
            else int(np.sqrt(n_features))
        )
        feat_idxs = np.random.choice(n_features, k, replace=False)

        # Find best split among sampled features
        best_feat, best_thresh = self._best_split(X, y, feat_idxs)

        # If no split improves Gini gain, make leaf node
        if best_feat is None:
            leaf_val = np.bincount(y).argmax()
            return DecisionNode(value=leaf_val)

        # Split data recursively
        left_idxs = np.where(X[:, best_feat] <= best_thresh)[0]
        right_idxs = np.where(X[:, best_feat] > best_thresh)[0]

        left_child = self._build_tree(X[left_idxs], y[left_idxs], depth + 1)
        right_child = self._build_tree(X[right_idxs], y[right_idxs], depth + 1)

        return DecisionNode(
            feature=best_feat,
            threshold=best_thresh,
            left=left_child,
            right=right_child,
        )

    def fit(self, X, y):
        self.root = self._build_tree(X, y)

    def _predict_row(self, node, x):
        if node.is_leaf():
            return node.value

        if x[node.feature] <= node.threshold:
            return self._predict_row(node.left, x)
        return self._predict_row(node.right, x)

    def predict(self, X):
        return np.array([self._predict_row(self.root, x) for x in X])


class RandomForestClassifier:
    """Random Forest Ensemble Classifier."""

    def __init__(
        self,
        n_trees=10,
        max_depth=10,
        min_samples_split=2,
        max_features=None,
    ):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.trees = []

    def _bootstrap_samples(self, X, y):
        """Draws N samples with replacement."""
        n_samples = X.shape[0]
        idxs = np.random.choice(n_samples, size=n_samples, replace=True)
        return X[idxs], y[idxs]

    def fit(self, X, y):
        self.trees = []
        n_features = X.shape[1]

        # Calculate k = floor(sqrt(d)) if not specified
        k_features = (
            self.max_features
            if self.max_features is not None
            else int(np.floor(np.sqrt(n_features)))
        )

        for _ in range(self.n_trees):
            tree = DecisionTreeClassifier(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                max_features=k_features,
            )
            # 1. Bootstrapping
            X_sample, y_sample = self._bootstrap_samples(X, y)
            # 2. Fit Tree
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)

    def predict(self, X):
        # Gather predictions from all trees: shape (n_trees, n_samples)
        tree_preds = np.array([tree.predict(X) for tree in self.trees])

        # Majority Vote across trees (along axis 0)
        # Transpose to shape (n_samples, n_trees) to compute mode per row
        tree_preds = tree_preds.T
        y_pred = [np.bincount(row).argmax() for row in tree_preds]

        return np.array(y_pred)

In [ ]:

X_train = np.array(
    [
        [2.0, 5.0, 1.0, 0.5],
        [7.0, 1.0, 4.0, 2.5],
        [8.0, 4.0, 3.0, 1.5],
        [1.0, 8.0, 2.0, 0.1],
    ]
)
y_train = np.array([0, 1, 1, 0])

# Unseen test instance x* = [6.0, 2.0, 3.5, 0.4]
X_test = np.array([[6.0, 2.0, 3.5, 0.4]])

# Instantiate and fit forest
np.random.seed(42)  
forest = RandomForestClassifier(n_trees=3, max_depth=3)
forest.fit(X_train, y_train)

# Predict
prediction = forest.predict(X_test)
print(f"Forest Prediction for test vector: {prediction[0]}")

Forest Prediction for test vector: 1


# Regression

The exact same architecture works for Random Forest Regression—you only swap out the classification components for regression math:

|Component|Classification|Regression|
|---|---|---|
|Node Impurity|Gini Impurity $G(S)$ or Entropy|Variance $\text{Var}(S)$ or Mean Squared Error (MSE)|
|Split Evaluation|Information Gain $\Delta G$|Variance Reduction $\Delta \text{Var}$|
|Leaf Output|Majority Vote Class (Mode)|Average Value (Mean $\bar{y}$)|
|Forest Aggregation|Majority Voting across trees|Average of Tree Predictions (Mean of $\hat{y}_m$)|

## Python Code

In [3]:
import numpy as np


class DecisionNode:
    """Represents a node or leaf in a decision tree for regression."""

    def __init__(
        self,
        feature=None,
        threshold=None,
        left=None,
        right=None,
        *,
        value=None,
    ):
        self.feature = feature  # Index of feature to split on
        self.threshold = threshold  # Threshold value for split
        self.left = left  # Left child node (x <= threshold)
        self.right = right  # Right child node (x > threshold)
        self.value = value  # Continuous target mean if leaf node

    def is_leaf(self):
        return self.value is not None


class DecisionTreeRegressor:
    """A single Decision Tree Regressor using Variance Reduction."""

    def __init__(self, max_depth=10, min_samples_split=2, max_features=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features  # k features to consider per split
        self.root = None

    def _variance(self, y):
        """Calculates Variance: Var(y) = (1/N) * sum((y_i - mean(y))^2)"""
        if len(y) == 0:
            return 0.0
        return np.var(y)

    def _variance_reduction(self, y, y_left, y_right):
        """Variance Reduction = Var_parent - (w_L * Var_left + w_R * Var_right)"""
        parent_var = self._variance(y)
        n = len(y)
        n_l, n_r = len(y_left), len(y_right)

        if n_l == 0 or n_r == 0:
            return 0.0

        child_var = (n_l / n) * self._variance(y_left) + (
            n_r / n
        ) * self._variance(y_right)
        return parent_var - child_var

    def _best_split(self, X, y, feat_idxs):
        """Finds feature & threshold yielding maximum Variance Reduction."""
        best_red = -1.0
        split_feat, split_thresh = None, None

        for feat in feat_idxs:
            X_column = X[:, feat]
            unique_vals = np.sort(np.unique(X_column))

            # Candidate thresholds as midpoints between adjacent sorted values
            thresholds = (unique_vals[:-1] + unique_vals[1:]) / 2.0

            for thresh in thresholds:
                left_idxs = np.where(X_column <= thresh)[0]
                right_idxs = np.where(X_column > thresh)[0]

                if len(left_idxs) == 0 or len(right_idxs) == 0:
                    continue

                var_red = self._variance_reduction(
                    y, y[left_idxs], y[right_idxs]
                )

                if var_red > best_red:
                    best_red = var_red
                    split_feat = feat
                    split_thresh = thresh

        return split_feat, split_thresh

    def _build_tree(self, X, y, depth=0):
        n_samples, n_features = X.shape

        # Stopping criteria: zero variance, max depth, or small sample size
        if (
            depth >= self.max_depth
            or np.var(y) == 0
            or n_samples < self.min_samples_split
        ):
            return DecisionNode(value=np.mean(y))

        # Feature Subsampling: select k random features at THIS split
        k = (
            self.max_features
            if self.max_features is not None
            else max(1, int(n_features / 3.0))
        )
        feat_idxs = np.random.choice(n_features, k, replace=False)

        # Find best split among sampled features
        best_feat, best_thresh = self._best_split(X, y, feat_idxs)

        # If no split reduces variance, return leaf node with mean value
        if best_feat is None:
            return DecisionNode(value=np.mean(y))

        # Recursively construct left and right subtrees
        left_idxs = np.where(X[:, best_feat] <= best_thresh)[0]
        right_idxs = np.where(X[:, best_feat] > best_thresh)[0]

        left_child = self._build_tree(X[left_idxs], y[left_idxs], depth + 1)
        right_child = self._build_tree(X[right_idxs], y[right_idxs], depth + 1)

        return DecisionNode(
            feature=best_feat,
            threshold=best_thresh,
            left=left_child,
            right=right_child,
        )

    def fit(self, X, y):
        self.root = self._build_tree(X, y)

    def _predict_row(self, node, x):
        if node.is_leaf():
            return node.value

        if x[node.feature] <= node.threshold:
            return self._predict_row(node.left, x)
        return self._predict_row(node.right, x)

    def predict(self, X):
        return np.array([self._predict_row(self.root, x) for x in X])


class RandomForestRegressor:
    """Random Forest Ensemble Regressor."""

    def __init__(
        self,
        n_trees=10,
        max_depth=10,
        min_samples_split=2,
        max_features=None,
    ):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.trees = []

    def _bootstrap_samples(self, X, y):
        """Draws N samples with replacement."""
        n_samples = X.shape[0]
        idxs = np.random.choice(n_samples, size=n_samples, replace=True)
        return X[idxs], y[idxs]

    def fit(self, X, y):
        self.trees = []
        n_features = X.shape[1]

        # For regression, standard feature subsampling size is d / 3
        k_features = (
            self.max_features
            if self.max_features is not None
            else max(1, int(np.floor(n_features / 3.0)))
        )

        for _ in range(self.n_trees):
            tree = DecisionTreeRegressor(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                max_features=k_features,
            )
            # Bootstrapping (Bagging)
            X_sample, y_sample = self._bootstrap_samples(X, y)
            # Fit individual tree
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)

    def predict(self, X):
        # Shape: (n_trees, n_samples)
        tree_preds = np.array([tree.predict(X) for tree in self.trees])

        # Average prediction across all trees (axis 0)
        return np.mean(tree_preds, axis=0)

In [4]:

X_train = np.array(
    [
        [850, 2, 10],
        [1200, 3, 5],
        [1500, 3, 20],
        [1800, 4, 2],
        [2200, 4, 15],
        [2500, 5, 8],
    ]
)
# Target values (continuous)
y_train = np.array([180.0, 250.0, 280.0, 360.0, 400.0, 480.0])

# Test sample
X_test = np.array([[1600, 3, 10]])

# Fit ensemble regressor
np.random.seed(42)
regressor = RandomForestRegressor(n_trees=5, max_depth=4)
regressor.fit(X_train, y_train)

prediction = regressor.predict(X_test)
print(f"Predicted Continuous Value: ${prediction[0]:.2f}k")

Predicted Continuous Value: $251.00k
